# Agentic Systematic Review — notebook d'exécution (Kaggle)
**À chaque nouvelle session :** exécuter les cellules 1 → 6, puis la section du lot en cours.
Tout le travail est sauvegardé dans `runs/` et poussé sur GitHub avec `sync()` → une coupure ne fait rien perdre.

Prérequis Kaggle : *Settings → Internet ON* · *Add-ons → Secrets* : `LLM_API_KEY`, `GITHUB_TOKEN` (cocher **attach**).

In [ ]:
# ===== 1. CONFIGURATION =====
GITHUB_USER  = "mouadili-merouane"
GITHUB_REPO  = "agentic-systematic-review"
GIT_EMAIL    = "ton.email@exemple.com"     # commits + 'polite pool' OpenAlex/NCBI

LLM_PROVIDER = "groq"
# Chaque rôle a une CHAÎNE de modèles : quand le quota journalier d'un modèle est atteint (200k tokens/jour
# par modèle en gratuit chez Groq), le pipeline bascule tout seul sur le suivant. Le travail fait est conservé.
MODELS_FAST   = ["openai/gpt-oss-20b", "qwen/qwen3.6-27b", "openai/gpt-oss-120b"]   # screener A
MODELS_STRONG = ["openai/gpt-oss-120b", "qwen/qwen3.6-27b", "openai/gpt-oss-20b"]   # arbitre, texte intégral, extraction, synthèse, critique
MODELS_B      = MODELS_STRONG                                                        # screener B

PROTOCOL     = "configs/protocol_demo.yaml"
HITL         = True                        # pauses de validation humaine

In [ ]:
# ===== 2. INSTALLATION =====
%pip install -q -U langgraph langgraph-checkpoint-sqlite langchain langchain-community langchain-groq \
    faiss-cpu rank_bm25 sentence-transformers pymupdf rapidfuzz "pydantic>=2" pyyaml "pillow<13" "tenacity<9.2" pypandoc_binary

In [ ]:
# ===== 3. SECRETS =====
import os
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
os.environ[{"groq": "GROQ_API_KEY", "google_genai": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY"}[LLM_PROVIDER]] = sec.get_secret("LLM_API_KEY")
os.environ["GITHUB_TOKEN"] = sec.get_secret("GITHUB_TOKEN")
try:    S2_KEY = sec.get_secret("S2_API_KEY")
except Exception: S2_KEY = None
print("✔ secrets chargés")

In [ ]:
# ===== 4. RESTAURATION DEPUIS GITHUB =====
import subprocess, pathlib, sys
WORK = pathlib.Path("/kaggle/working"); REPO = WORK / GITHUB_REPO
remote = f"https://{os.environ['GITHUB_TOKEN']}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"

def sh(cmd, cwd=REPO):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(os.environ["GITHUB_TOKEN"], "***")
    if r.returncode: raise RuntimeError(out)
    return out

print(sh(f"git clone {remote} {REPO}", cwd=WORK) if not REPO.exists() else sh("git pull --rebase"))
sh(f'git config user.email "{GIT_EMAIL}"'); sh(f'git config user.name "{GITHUB_USER}"')
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
os.chdir(REPO)

def sync(msg="checkpoint"):
    """Sauvegarde code + résultats sur GitHub. À lancer après chaque étape longue."""
    sh("git add -A")
    if not sh("git status --porcelain").strip(): return print("rien à sauvegarder")
    sh(f'git commit -m "{msg}"'); sh("git pull --rebase"); sh("git push")
    print("✔ poussé sur GitHub :", msg)

In [ ]:
# ===== 5. LLM + CACHE + CONTEXTE =====
import yaml
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from sr_agent.llm import FallbackLLM
from sr_agent.utils.checkpoint import RunStore
from sr_agent.graph import Ctx, build_graph, drive

cfg = yaml.safe_load(open(PROTOCOL))
RUN_DIR = REPO / "runs" / cfg["review_id"]; RUN_DIR.mkdir(parents=True, exist_ok=True)
set_llm_cache(SQLiteCache(database_path=str(REPO / "runs" / "llm_cache.db")))

store = RunStore(RUN_DIR)
on_switch = lambda old, new, err: store.log("coordinator", "model_switch", old=old, new=new, reason=err)
llm_fast   = FallbackLLM(MODELS_FAST, LLM_PROVIDER, on_switch)
llm_strong = FallbackLLM(MODELS_STRONG, LLM_PROVIDER, on_switch)
llm_b      = llm_strong if MODELS_B == MODELS_STRONG else FallbackLLM(MODELS_B, LLM_PROVIDER, on_switch)
store.log("coordinator", "session_start", provider=LLM_PROVIDER, fast=MODELS_FAST, strong=MODELS_STRONG, b=MODELS_B)
ctx = Ctx(store=store, cfg=cfg, llm_fast=llm_fast, llm_strong=llm_strong, llm_b=llm_b,
          email=GIT_EMAIL, s2_key=S2_KEY, hitl=HITL, sync=sync,
          extra={"models": {"screener_a": MODELS_FAST[0], "screener_b": MODELS_B[0], "strong": " / ".join(MODELS_STRONG)}})
print("Revue :", cfg["review_id"], "| fichiers existants :", sorted(p.name for p in RUN_DIR.iterdir()))

In [ ]:
# ===== 6. TESTS RAPIDES (LLM + bases) =====
from sr_agent.llm import call
for n, m in [("fast", llm_fast), ("strong", llm_strong)]:
    try: print(f"LLM {n:6s} ✔", m.current, "→", call(m, "Answer with one word.", "Say OK").strip()[:20])
    except Exception as e: print(f"LLM {n:6s} ✘", str(e)[:200])

from sr_agent.tools.sources import SEARCHERS
for s in SEARCHERS:
    try:
        kw = {"api_key": S2_KEY} if s == "semantic_scholar" else {}
        print(f"{s:17s} ✔", len(SEARCHERS[s]("heart failure machine learning", [2020, 2026], 3, **kw)), "résultats")
    except Exception as e: print(f"{s:17s} ✘", str(e)[:150])

---
## EXÉCUTION DU PIPELINE
Une seule cellule : le coordinateur reprend là où il s'était arrêté (les étapes faites sont sautées, sans appel LLM).

Pauses humaines : protocole → stratégie → screening titre/résumé → screening texte intégral.
Répondre `ok`, ou une correction en langage naturel (protocole/stratégie), ou `include <id>` / `exclude <id>` (screening).

💡 Activer le GPU **T4** accélère la construction de l'index RAG (embeddings) ; optionnel.

In [ ]:
graph = build_graph(ctx, str(RUN_DIR / "graph_state.sqlite"))
try:
    result = drive(graph, thread_id=cfg["review_id"])
finally:
    sync("pipeline : sauvegarde de fin de lot")   # sauvegarde même en cas d'erreur / quota

In [ ]:
# ===== Résultats du lot 1 =====
import json, pandas as pd
print(json.dumps(store.load_json("prisma_counts.json"), indent=2))
print(json.dumps(store.load_json("02_search_strategy.json")["queries"], indent=2, ensure_ascii=False))
df = pd.DataFrame(store.load_json("04_records.json", []))
df[["title", "year", "venue", "found_in"]].head(15)

## Résultats du lot 2 — screening & texte intégral

In [ ]:
print(json.dumps(store.load_json("05_ta_summary.json"), indent=2))
ft_ = {r["id"]: r for r in store.read_jsonl("07_ft.jsonl")}
recs_ = {r["id"]: r for r in store.load_json("04_records.json")}
inc_ = pd.DataFrame([{"id": i, "title": recs_[i]["title"][:100], "year": recs_[i]["year"], "basis": d.get("basis"),
                      "rationale": d["rationale"][:150]} for i, d in ft_.items() if d["decision"] == "include"])
print(len(inc_), "études incluses"); inc_

## Résultats du lot 3 — extraction, synthèse, contrôle qualité, rapport

In [ ]:
from IPython.display import Image, Markdown, display
crit = store.load_json("10_critique.json", {})
print("Contrôle qualité :", json.dumps(crit.get("checks", {}).get("evidence_verification_rate")), "des citations vérifiées |",
      "révisions :", crit.get("rounds"), "|", crit.get("review", {}).get("overall_assessment", ""))
display(Image(str(RUN_DIR / "report" / "prisma_flow.png"), width=650))
display(Markdown((RUN_DIR / "report" / "manuscript.md").read_text()[:6000]))